# Case BI Connect/Virtueyes - Pipeline em camadas

**Mário Schenkel**  
Data Analyst | Analytics Engineer

Este notebook prepara os dados usados no dashboard.

Ele cria e salva tres camadas de dados:

1. **Stage**: leitura dos arquivos brutos da Anatel com padronizacao minima.
2. **Quality**: filtro M2M, regras de qualidade e bases validadas.
3. **Analytics**: marts finais para Metabase e analises executivas.

A ideia e deixar o processamento facil de conferir.

## 0. Atualizacao das seeds

Esta etapa baixa os dados brutos da Anatel e salva os arquivos em `seeds/`.

Por padrao o download fica desligado para evitar baixar arquivos grandes sem querer. Para atualizar a base, altere `BAIXAR_DADOS_ANATEL` para `True` e execute a celula.

In [ ]:
from pathlib import Path
import re
from urllib.request import urlretrieve
from zipfile import ZipFile

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SEEDS_DIR = PROJECT_ROOT / "seeds"
DENSITIES_DIR = SEEDS_DIR / "densidades"
DOWNLOADS_DIR = SEEDS_DIR / "_downloads"

for path in [SEEDS_DIR, DENSITIES_DIR, DOWNLOADS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BAIXAR_DADOS_ANATEL = False
FORCAR_DOWNLOAD = False
ANO_MINIMO_ACESSOS = 2025

ACESSOS_URL = "https://www.anatel.gov.br/dadosabertos/paineis_de_dados/acessos/acessos_telefonia_movel.zip"


def arquivo_acessos_valido(nome_arquivo):
    return re.fullmatch(r"Acessos_Telefonia_Movel_\d{4}_[12]S\.csv", nome_arquivo) is not None


def ano_arquivo_acessos(nome_arquivo):
    match = re.search(r"Acessos_Telefonia_Movel_(\d{4})_[12]S\.csv", nome_arquivo)
    return int(match.group(1)) if match else None


def arquivo_acessos_no_periodo(nome_arquivo):
    ano = ano_arquivo_acessos(nome_arquivo)
    return ano is not None and (ANO_MINIMO_ACESSOS is None or ano >= ANO_MINIMO_ACESSOS)


def limpar_seeds_acessos_fora_do_periodo():
    removidos = []
    for arquivo in SEEDS_DIR.glob("Acessos_Telefonia_Movel_*.csv"):
        if not arquivo_acessos_valido(arquivo.name) or not arquivo_acessos_no_periodo(arquivo.name):
            arquivo.unlink()
            removidos.append(arquivo.name)

    if removidos:
        print("Arquivos removidos de seeds/:")
        for nome in removidos:
            print(f"- {nome}")
    else:
        print("Nenhum arquivo antigo para remover de seeds/.")


def baixar_arquivo(url, destino, forcar=False):
    if destino.exists() and not forcar:
        print(f"Arquivo ja existe: {destino.name}")
        return destino

    print(f"Baixando {url}")
    urlretrieve(url, destino)
    print(f"Salvo em {destino}")
    return destino


def destino_csv_anatel(nome_arquivo):
    nome = nome_arquivo.lower()
    if "densidade" in nome:
        return DENSITIES_DIR / nome_arquivo
    if arquivo_acessos_valido(nome_arquivo) and arquivo_acessos_no_periodo(nome_arquivo):
        return SEEDS_DIR / nome_arquivo
    return None


def extrair_csvs_anatel(zip_path):
    extraidos = []
    with ZipFile(zip_path) as zf:
        for item in zf.infolist():
            if item.is_dir() or not item.filename.lower().endswith(".csv"):
                continue

            nome = Path(item.filename).name
            saida = destino_csv_anatel(nome)
            if saida is None:
                print(f"Ignorado: {nome}")
                continue
            saida.parent.mkdir(parents=True, exist_ok=True)
            if saida.exists() and not FORCAR_DOWNLOAD:
                print(f"CSV ja existe: {saida.name}")
                extraidos.append(saida)
                continue

            with zf.open(item) as origem, saida.open("wb") as alvo:
                alvo.write(origem.read())
            print(f"Extraido: {saida}")
            extraidos.append(saida)

    return extraidos


limpar_seeds_acessos_fora_do_periodo()

if BAIXAR_DADOS_ANATEL:
    acessos_zip = baixar_arquivo(
        ACESSOS_URL,
        DOWNLOADS_DIR / "acessos_telefonia_movel.zip",
        FORCAR_DOWNLOAD,
    )
    extrair_csvs_anatel(acessos_zip)
else:
    print("Download desligado. Para atualizar as seeds, altere BAIXAR_DADOS_ANATEL para True.")

## 1. Setup do ambiente

Nesta etapa ficam os caminhos, bibliotecas e funcoes auxiliares.

Os arquivos brutos ficam em `seeds/`.

In [ ]:
from pathlib import Path
import json
import re
import shutil

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SEEDS_DIR = PROJECT_ROOT / "seeds"
DENSITIES_DIR = SEEDS_DIR / "densidades"
ANO_MINIMO_ACESSOS = 2025

DATA_DIR = PROJECT_ROOT / "data"
STAGE_DIR = DATA_DIR / "01_stage"
QUALITY_DIR = DATA_DIR / "02_quality"
ANALYTICS_DIR = DATA_DIR / "03_analytics"
MARTS_DIR = ANALYTICS_DIR / "marts"
EXPORTS_DIR = ANALYTICS_DIR / "exports"
REPORTS_DIR = DATA_DIR / "reports"

for path in [STAGE_DIR, QUALITY_DIR, ANALYTICS_DIR, MARTS_DIR, EXPORTS_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def sql_literal(value):
    text = str(value).replace("\\", "/").replace("'", "''")
    return f"'{text}'"

def files_as_sql_list(files):
    return "[" + ", ".join(sql_literal(path) for path in files) + "]"

acessos_files = sorted(
    file for file in SEEDS_DIR.glob("Acessos_Telefonia_Movel_*.csv")
    if re.fullmatch(r"Acessos_Telefonia_Movel_\d{4}_[12]S\.csv", file.name)
    and (ANO_MINIMO_ACESSOS is None or int(file.name.split("_")[3]) >= ANO_MINIMO_ACESSOS)
)
densidade_files = sorted(DENSITIES_DIR.glob("*.csv"))

if not acessos_files:
    raise FileNotFoundError("Nenhum arquivo Acessos_Telefonia_Movel_*.csv encontrado em seeds/")

print("Arquivos de acessos encontrados:")
for file in acessos_files:
    print(f"- {file.name} ({file.stat().st_size / 1024 / 1024:,.2f} MB)")

print("\nArquivos de densidade encontrados:")
for file in densidade_files:
    print(f"- {file.name} ({file.stat().st_size / 1024 / 1024:,.2f} MB)")

con = duckdb.connect()

## 2. Camada Stage

Objetivo da stage:

- ler os CSVs grandes da Anatel;
- normalizar nomes de colunas;
- converter tipos essenciais;
- salvar uma copia estruturada em Parquet;
- preservar granularidade e produtos originais.

Aqui ainda **nao** filtramos M2M. A stage representa a fonte bruta organizada.

In [ ]:
acessos_scan = (
    "read_csv_auto("
    f"{files_as_sql_list(acessos_files)}, "
    "delim=';', header=true, union_by_name=true, normalize_names=true, "
    "filename=true, ignore_errors=false)"
)

stage_acessos_path = STAGE_DIR / "stg_acessos_movel.parquet"

con.execute(
    f"""
    copy (
        select
            cast(ano as integer) as ano,
            cast(mes as integer) as mes,
            cast(cast(ano as varchar) || '-' || lpad(cast(mes as varchar), 2, '0') || '-01' as date) as periodo,
            trim(grupo_economico) as grupo_economico,
            trim(empresa) as empresa,
            lpad(regexp_replace(cast(cnpj as varchar), '[^0-9]', '', 'g'), 14, '0') as cnpj,
            trim(porte_da_prestadora) as porte_da_prestadora,
            trim(uf) as uf,
            trim(municipio) as municipio,
            cast(codigo_ibge_municipio as integer) as codigo_ibge_municipio,
            cast(codigo_nacional as integer) as codigo_nacional,
            cast(codigo_nacional_chip as integer) as codigo_nacional_chip,
            trim(modalidade_de_cobranca) as modalidade_de_cobranca,
            trim(tecnologia) as tecnologia,
            trim(tecnologia_geracao) as tecnologia_geracao,
            trim(tipo_de_pessoa) as tipo_de_pessoa,
            trim(tipo_de_produto) as tipo_de_produto,
            coalesce(try_cast(acessos as bigint), 0) as acessos,
            filename as arquivo_origem
        from {acessos_scan}
    )
    to {sql_literal(stage_acessos_path)}
    (format parquet, compression zstd)
    """
)

stage_produtos = con.sql(
    f"""
    select
        tipo_de_produto,
        count(*) as linhas,
        sum(acessos) as acessos
    from read_parquet({sql_literal(stage_acessos_path)})
    group by 1
    order by linhas desc
    """
).df()

stage_produtos.to_csv(REPORTS_DIR / "stage_resumo_produtos.csv", index=False, sep=";")

print(f"Stage de acessos salva em: {stage_acessos_path}")
print("\nResumo por tipo de produto:")
print(stage_produtos.to_string(index=False))

In [ ]:
stage_densidade_path = STAGE_DIR / "stg_densidade_movel.parquet"

if densidade_files:
    densidade_scan = (
        "read_csv_auto("
        f"{files_as_sql_list(densidade_files)}, "
        "delim=';', header=true, union_by_name=true, normalize_names=true, "
        "decimal_separator=',', filename=true, ignore_errors=false)"
    )

    con.execute(
        f"""
        copy (
            select
                cast(ano as integer) as ano,
                cast(mes as integer) as mes,
                cast(cast(ano as varchar) || '-' || lpad(cast(mes as varchar), 2, '0') || '-01' as date) as periodo,
                trim(uf) as uf,
                trim(municipio) as municipio,
                trim(codigo_ibge) as codigo_ibge,
                cast(densidade as double) as densidade,
                trim(nivel_geografico_densidade) as nivel_geografico_densidade,
                filename as arquivo_origem
            from {densidade_scan}
        )
        to {sql_literal(stage_densidade_path)}
        (format parquet, compression zstd)
        """
    )

    stage_densidade = con.sql(
        f"""
        select
            nivel_geografico_densidade,
            count(*) as linhas,
            min(periodo) as periodo_min,
            max(periodo) as periodo_max
        from read_parquet({sql_literal(stage_densidade_path)})
        group by 1
        order by linhas desc
        """
    ).df()
    stage_densidade.to_csv(REPORTS_DIR / "stage_resumo_densidade.csv", index=False, sep=";")
    print(f"Stage de densidade salva em: {stage_densidade_path}")
    print(stage_densidade.to_string(index=False))
else:
    print("Base de densidade nao encontrada. A camada de oportunidades por UF sera criada sem densidade.")

## 3. Camada Quality

Objetivo da quality:

- aplicar a regra obrigatoria do case: `Tipo de Produto = M2M`;
- manter somente dados prontos para analise;
- executar checks de qualidade;
- salvar relatorios para auditoria.

O dashboard usa a base ja filtrada, nao o CSV bruto.

In [ ]:
quality_m2m_path = QUALITY_DIR / "acessos_m2m_quality.parquet"

con.execute(
    f"""
    copy (
        select *
        from read_parquet({sql_literal(stage_acessos_path)})
        where tipo_de_produto = 'M2M'
    )
    to {sql_literal(quality_m2m_path)}
    (format parquet, compression zstd)
    """
)

quality_densidade_uf_path = QUALITY_DIR / "densidade_uf_quality.parquet"

if stage_densidade_path.exists():
    con.execute(
        f"""
        copy (
            select *
            from read_parquet({sql_literal(stage_densidade_path)})
            where nivel_geografico_densidade = 'UF'
        )
        to {sql_literal(quality_densidade_uf_path)}
        (format parquet, compression zstd)
        """
    )

checks = []

def add_check(nome, passou, valor, detalhe):
    checks.append({
        "check": nome,
        "status": "OK" if passou else "FAIL",
        "valor": valor,
        "detalhe": detalhe,
    })

total_m2m = con.sql(f"select count(*) from read_parquet({sql_literal(quality_m2m_path)})").fetchone()[0]
add_check("base_m2m_possui_linhas", total_m2m > 0, total_m2m, "Base quality M2M nao pode ficar vazia.")

nao_m2m = con.sql(
    f"select count(*) from read_parquet({sql_literal(quality_m2m_path)}) where tipo_de_produto <> 'M2M'"
).fetchone()[0]
add_check("somente_tipo_produto_m2m", nao_m2m == 0, nao_m2m, "Todos os registros devem ser M2M.")

negativos = con.sql(
    f"select count(*) from read_parquet({sql_literal(quality_m2m_path)}) where acessos < 0"
).fetchone()[0]
add_check("acessos_nao_negativos", negativos == 0, negativos, "Acessos nao podem ser negativos.")

periodos_nulos = con.sql(
    f"select count(*) from read_parquet({sql_literal(quality_m2m_path)}) where ano is null or mes is null or periodo is null"
).fetchone()[0]
add_check("periodos_validos", periodos_nulos == 0, periodos_nulos, "Ano, mes e periodo devem estar preenchidos.")

connect_linhas = con.sql(
    f"select count(*) from read_parquet({sql_literal(quality_m2m_path)}) where lower(empresa) like '%connect%'"
).fetchone()[0]
add_check("connect_presente", connect_linhas > 0, connect_linhas, "Connect precisa existir na base M2M.")

if quality_densidade_uf_path.exists():
    densidade_negativa = con.sql(
        f"select count(*) from read_parquet({sql_literal(quality_densidade_uf_path)}) where densidade < 0"
    ).fetchone()[0]
    add_check("densidade_nao_negativa", densidade_negativa == 0, densidade_negativa, "Densidade nao pode ser negativa.")

checks_df = pd.DataFrame(checks)
checks_df.to_csv(REPORTS_DIR / "quality_checks.csv", index=False, sep=";")

resumo_quality = {
    "linhas_m2m": int(total_m2m),
    "linhas_connect": int(connect_linhas),
    "checks_ok": int((checks_df["status"] == "OK").sum()),
    "checks_fail": int((checks_df["status"] == "FAIL").sum()),
}
(REPORTS_DIR / "quality_resumo.json").write_text(json.dumps(resumo_quality, indent=2), encoding="utf-8")

print("Checks de qualidade:")
print(checks_df.to_string(index=False))

if (checks_df["status"] == "FAIL").any():
    raise ValueError("Existem checks de qualidade com falha. Corrija antes de seguir para analytics.")

In [ ]:
quality_mes = con.sql(
    f"""
    select
        periodo,
        count(*) as linhas,
        sum(acessos) as acessos_m2m
    from read_parquet({sql_literal(quality_m2m_path)})
    group by 1
    order by 1
    """
).df()
quality_mes.to_csv(REPORTS_DIR / "quality_resumo_m2m_mes.csv", index=False, sep=";")

quality_connect = con.sql(
    f"""
    select
        periodo,
        empresa,
        sum(acessos) as acessos_connect
    from read_parquet({sql_literal(quality_m2m_path)})
    where lower(empresa) like '%connect%'
    group by 1, 2
    order by 1
    """
).df()
quality_connect.to_csv(REPORTS_DIR / "quality_connect_mes.csv", index=False, sep=";")

print("Resumo M2M por mes:")
print(quality_mes.to_string(index=False))
print("\nConnect por mes:")
print(quality_connect.to_string(index=False))

## 4. Camada Analytics

Objetivo da analytics:

- criar tabelas de negocio prontas para dashboard;
- calcular crescimento, ranking e market share;
- combinar oportunidade regional com densidade movel;
- salvar um DuckDB final e exports para auditoria.

Esta e a camada usada pelo Metabase.

In [ ]:
analytics_db_path = ANALYTICS_DIR / "case_bi.duckdb"
if analytics_db_path.exists():
    analytics_db_path.unlink()

analytics_con = duckdb.connect(str(analytics_db_path))

analytics_con.execute(
    f"""
    create table stg_acessos_m2m as
    select *
    from read_parquet({sql_literal(quality_m2m_path)})
    """
)

if quality_densidade_uf_path.exists():
    analytics_con.execute(
        f"""
        create table stg_densidade_uf as
        select *
        from read_parquet({sql_literal(quality_densidade_uf_path)})
        """
    )
else:
    analytics_con.execute(
        """
        create table stg_densidade_uf (
            ano integer,
            mes integer,
            periodo date,
            uf varchar,
            municipio varchar,
            codigo_ibge varchar,
            densidade double,
            nivel_geografico_densidade varchar,
            arquivo_origem varchar
        )
        """
    )

print(f"DuckDB analytics criado em: {analytics_db_path}")

In [ ]:
analytics_con.execute(
    """
    create table mart_m2m_empresa_mes as
    with agregado as (
        select
            periodo,
            ano,
            mes,
            grupo_economico,
            empresa,
            cnpj,
            porte_da_prestadora,
            sum(acessos) as acessos
        from stg_acessos_m2m
        group by 1, 2, 3, 4, 5, 6, 7
    ),
    enriquecido as (
        select
            *,
            sum(acessos) over (partition by periodo) as acessos_mercado,
            rank() over (partition by periodo order by acessos desc) as ranking_acessos,
            lag(acessos) over (partition by empresa, cnpj order by periodo) as acessos_mes_anterior
        from agregado
    )
    select
        *,
        acessos - coalesce(acessos_mes_anterior, 0) as crescimento_abs,
        case
            when acessos_mes_anterior > 0
                then (acessos - acessos_mes_anterior) * 1.0 / acessos_mes_anterior
            else null
        end as crescimento_pct,
        case
            when acessos_mercado > 0 then acessos * 1.0 / acessos_mercado
            else null
        end as market_share,
        case when lower(empresa) like '%connect%' then true else false end as is_connect
    from enriquecido
    """
)

analytics_con.execute(
    """
    create table mart_m2m_empresa_uf_mes as
    with agregado as (
        select
            periodo,
            ano,
            mes,
            empresa,
            cnpj,
            uf,
            sum(acessos) as acessos
        from stg_acessos_m2m
        group by 1, 2, 3, 4, 5, 6
    )
    select
        *,
        lag(acessos) over (partition by empresa, cnpj, uf order by periodo) as acessos_mes_anterior,
        acessos - coalesce(lag(acessos) over (partition by empresa, cnpj, uf order by periodo), 0) as crescimento_abs
    from agregado
    """
)

analytics_con.execute(
    """
    create table mart_m2m_tecnologia_mes as
    select
        periodo,
        ano,
        mes,
        empresa,
        cnpj,
        tecnologia,
        tecnologia_geracao,
        tipo_de_pessoa,
        sum(acessos) as acessos
    from stg_acessos_m2m
    group by 1, 2, 3, 4, 5, 6, 7, 8
    """
)

print("Marts de empresa, UF e tecnologia criadas.")

In [ ]:
analytics_con.execute(
    """
    create table mart_oportunidade_uf_mes as
    with mercado_uf as (
        select
            periodo,
            ano,
            mes,
            uf,
            sum(acessos) as acessos_m2m,
            sum(case when lower(empresa) like '%connect%' then acessos else 0 end) as acessos_connect,
            count(distinct empresa || '|' || cnpj) as qtd_empresas
        from stg_acessos_m2m
        group by 1, 2, 3, 4
    ),
    densidade_uf as (
        select
            periodo,
            uf,
            avg(densidade) as densidade_movel
        from stg_densidade_uf
        group by 1, 2
    ),
    enriquecido as (
        select
            m.*,
            d.densidade_movel,
            lag(m.acessos_m2m) over (partition by m.uf order by m.periodo) as acessos_m2m_mes_anterior,
            lag(m.acessos_connect) over (partition by m.uf order by m.periodo) as acessos_connect_mes_anterior,
            sum(m.acessos_m2m) over (partition by m.periodo) as acessos_m2m_brasil
        from mercado_uf m
        left join densidade_uf d on m.periodo = d.periodo and m.uf = d.uf
    )
    select
        *,
        acessos_m2m - coalesce(acessos_m2m_mes_anterior, 0) as crescimento_m2m_abs,
        acessos_connect - coalesce(acessos_connect_mes_anterior, 0) as crescimento_connect_abs,
        case
            when acessos_m2m_brasil > 0 then acessos_m2m * 1.0 / acessos_m2m_brasil
            else null
        end as share_m2m_brasil,
        case when acessos_connect > 0 then true else false end as connect_presente,
        case
            when densidade_movel is null then 'Sem densidade'
            when densidade_movel < 90 and crescimento_m2m_abs > 0 then 'Baixa densidade e M2M crescendo'
            when densidade_movel >= 100 and crescimento_m2m_abs > 0 then 'Mercado maduro e M2M crescendo'
            when crescimento_m2m_abs > 0 then 'M2M crescendo'
            else 'Monitorar'
        end as leitura_oportunidade
    from enriquecido
    """
)

analytics_con.execute(
    """
    create table mart_radar_concorrentes as
    with limites as (
        select min(periodo) as periodo_inicio, max(periodo) as periodo_fim
        from mart_m2m_empresa_mes
    ),
    inicio as (
        select m.*
        from mart_m2m_empresa_mes m
        join limites l on m.periodo = l.periodo_inicio
    ),
    fim as (
        select m.*
        from mart_m2m_empresa_mes m
        join limites l on m.periodo = l.periodo_fim
    ),
    comparativo as (
        select
            coalesce(f.empresa, i.empresa) as empresa,
            coalesce(f.cnpj, i.cnpj) as cnpj,
            coalesce(f.grupo_economico, i.grupo_economico) as grupo_economico,
            coalesce(f.porte_da_prestadora, i.porte_da_prestadora) as porte_da_prestadora,
            i.acessos as acessos_inicio,
            f.acessos as acessos_fim,
            coalesce(f.acessos, 0) - coalesce(i.acessos, 0) as crescimento_periodo_abs,
            case
                when i.acessos > 0
                    then (coalesce(f.acessos, 0) - i.acessos) * 1.0 / i.acessos
                else null
            end as crescimento_periodo_pct,
            f.market_share as market_share_fim,
            f.ranking_acessos as ranking_fim,
            coalesce(f.is_connect, i.is_connect, false) as is_connect
        from inicio i
        full outer join fim f on i.empresa = f.empresa and i.cnpj = f.cnpj
    )
    select
        *,
        case
            when is_connect then 'Connect/Virtueyes'
            when ranking_fim <= 5 then 'Lideres de mercado'
            when crescimento_periodo_abs >= 100000 then 'Crescimento absoluto relevante'
            when crescimento_periodo_pct >= 0.25 and acessos_fim >= 10000 then 'Player agressivo'
            when acessos_fim >= 10000 then 'Monitorar'
            else 'Baixa prioridade'
        end as classificacao_radar
    from comparativo
    """
)

print("Marts de oportunidade e radar criadas.")

In [ ]:
marts = {
    "mart_m2m_empresa_mes": "Grao mensal por empresa, com ranking, crescimento e market share.",
    "mart_m2m_empresa_uf_mes": "Grao mensal por empresa e UF.",
    "mart_oportunidade_uf_mes": "Grao mensal por UF, combinando M2M, Connect e densidade movel.",
    "mart_m2m_tecnologia_mes": "Grao mensal por empresa, tecnologia e tipo de pessoa.",
    "mart_radar_concorrentes": "Comparativo inicio vs fim para priorizar concorrentes no radar.",
}

catalogo = []
for table, descricao in marts.items():
    parquet_path = MARTS_DIR / f"{table}.parquet"
    csv_path = EXPORTS_DIR / f"{table}.csv"

    analytics_con.execute(
        f"copy {table} to {sql_literal(parquet_path)} (format parquet, compression zstd)"
    )
    analytics_con.execute(
        f"copy {table} to {sql_literal(csv_path)} (header, delimiter ';')"
    )

    linhas = analytics_con.sql(f"select count(*) from {table}").fetchone()[0]
    catalogo.append({
        "tabela": table,
        "linhas": int(linhas),
        "descricao": descricao,
        "parquet": str(parquet_path.relative_to(PROJECT_ROOT)),
        "csv": str(csv_path.relative_to(PROJECT_ROOT)),
    })

catalogo_df = pd.DataFrame(catalogo)
catalogo_df.to_csv(ANALYTICS_DIR / "catalogo_marts.csv", index=False, sep=";")

print("Catalogo de marts:")
print(catalogo_df.to_string(index=False))

## 5. Leitura executiva inicial

Esta ultima parte gera tabelas pequenas para conferencia.

In [ ]:
connect_ultimo_mes = analytics_con.sql(
    """
    select
        periodo,
        empresa,
        acessos,
        crescimento_abs,
        round(crescimento_pct * 100, 2) as crescimento_pct,
        ranking_acessos,
        round(market_share * 100, 3) as market_share_pct
    from mart_m2m_empresa_mes
    where is_connect
    order by periodo desc
    limit 1
    """
).df()

top_crescimento = analytics_con.sql(
    """
    select
        empresa,
        acessos,
        crescimento_abs,
        round(crescimento_pct * 100, 2) as crescimento_pct,
        ranking_acessos
    from mart_m2m_empresa_mes
    where periodo = (select max(periodo) from mart_m2m_empresa_mes)
    order by crescimento_abs desc
    limit 10
    """
).df()

oportunidades_uf = analytics_con.sql(
    """
    select
        uf,
        acessos_m2m,
        crescimento_m2m_abs,
        acessos_connect,
        round(densidade_movel, 2) as densidade_movel,
        leitura_oportunidade
    from mart_oportunidade_uf_mes
    where periodo = (select max(periodo) from mart_oportunidade_uf_mes)
    order by crescimento_m2m_abs desc
    limit 12
    """
).df()

connect_ultimo_mes.to_csv(REPORTS_DIR / "insight_connect_ultimo_mes.csv", index=False, sep=";")
top_crescimento.to_csv(REPORTS_DIR / "insight_top_crescimento.csv", index=False, sep=";")
oportunidades_uf.to_csv(REPORTS_DIR / "insight_oportunidades_uf.csv", index=False, sep=";")

print("Connect no ultimo mes:")
print(connect_ultimo_mes.to_string(index=False))
print("\nTop crescimento absoluto no ultimo mes:")
print(top_crescimento.to_string(index=False))
print("\nOportunidades por UF:")
print(oportunidades_uf.to_string(index=False))

analytics_con.close()
con.close()

## 6. O que foi salvo

Depois de executar o notebook, os principais artefatos ficam em:

- `data/01_stage/`: dados brutos estruturados em Parquet.
- `data/02_quality/`: dados M2M validados e densidade UF validada.
- `data/03_analytics/case_bi.duckdb`: banco final para Metabase.
- `data/03_analytics/marts/`: marts em Parquet.
- `data/03_analytics/exports/`: marts em CSV para auditoria.
- `data/reports/`: checks, resumos e insights iniciais.

Essa separacao facilita conferir o que foi gerado.